# Random Forest Baseline for IoT Traffic Classification

**Goal.** Train a Random Forest that labels a network-traffic record as **benign** (`0`) or **malicious** (`1`),
and measure how well it does on data it has not seen.

### What is a Random Forest?

A *decision tree* classifies a record by asking a chain of yes/no questions about its features, such as
"is this packet count above some value?". One tree can overfit the training data. A **Random Forest**
trains many trees, each on a random resample of the rows and a random subset of features at every split,
then lets them vote. Averaging many different trees gives steadier predictions than any single tree.

### Run on Kaggle

Use a private Kaggle notebook with the reviewed repository checkout and the approved G3B data package attached. Follow the setup in the accompanying README. The shared runner verifies the package and uses TRAIN for fitting and VALIDATION for reporting.

## Configure the run

Set the paths below to your checkout and attached package. The default takes a seeded 5% sample within each TRAIN class, preserving class proportions. Subset runs are diagnostic. Use a fraction of 0.0001 for a quick runtime smoke.

In [ ]:
from pathlib import Path

SOURCE_ROOT = Path("/kaggle/working/iotguard")
PACKAGE_DIRECTORY = Path("/kaggle/input/YOUR-G3B-DATASET/g3b_primary_v1")
FRACTION = 0.05
JOBS = 2
TREES = 100
RUN_TRAINING = False

## Check source and package identity

This preflight only reads the package manifest and the checkout revision. At training time the runner also checks payload SHA-256 values before reading TRAIN and VALIDATION.

In [ ]:
import hashlib
import subprocess

EXPECTED_MANIFEST = (
    "6ab99e86397242e219bfa1f8490d8e03b169ec87a092af80cb3e7aef1e61147b"
)
manifest = PACKAGE_DIRECTORY / "manifest.json"
if not manifest.is_file():
    raise FileNotFoundError(f"Set PACKAGE_DIRECTORY to the approved G3B root: {manifest}")
if hashlib.sha256(manifest.read_bytes()).hexdigest() != EXPECTED_MANIFEST:
    raise ValueError("Unexpected G3B package manifest SHA-256")
revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=SOURCE_ROOT, text=True
).strip()
print("Source revision:", revision)
print("G3B manifest SHA-256:", EXPECTED_MANIFEST)

## Train Random Forest and evaluate on VALIDATION

Enable training in the configuration cell after checking the source revision and package. The shared runner enforces Kaggle-only fitting and writes its outputs to a fresh run directory. It records resolved settings, source revision, actual TRAIN membership, validation metrics and the package verification receipt.

In [ ]:
import sys
from datetime import UTC, datetime

if RUN_TRAINING:
    run_stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
    output = Path("/kaggle/working/rf-runs") / run_stamp
    ml_root = SOURCE_ROOT / "ml"
    subprocess.run(
        [
            sys.executable,
            str(ml_root / "scripts/train_rf_baseline.py"),
            "--package", str(PACKAGE_DIRECTORY),
            "--seeds", str(ml_root / "config/seeds.yaml"),
            "--out", str(output),
            "--fraction", str(FRACTION),
            "--jobs", str(JOBS),
            *["--trees", str(TREES)],
        ],
        cwd=SOURCE_ROOT,
        check=True,
    )
    print("Run artifacts:", output)
else:
    print("Preflight only. Set RUN_TRAINING = True to run on Kaggle.")

## Review

Use the runner's `result.json`, `run_notes.json` and `package_verification.json` for review. Keep the official comparison and the frozen final evaluation associated with their original run IDs.